# Doc2Vec で文書ベクトルを学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/07_word_vectors/doc2vec.ipynb)

NLP クックブック 07-10 ／ 単語ベクトル

文書そのものにベクトルを割り当てて、似た文書を探したい (word2vec の文書版)。

In [1]:
!pip install -q gensim

In [2]:
import nltk
nltk.download("reuters", quiet=True)
from nltk.corpus import reuters
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

ids = reuters.fileids()[:3000]
docs = [TaggedDocument([w.lower() for w in reuters.words(i) if w.isalpha()], [n]) for n, i in enumerate(ids)]
model = Doc2Vec(docs, vector_size=100, window=5, min_count=3, epochs=20, dm=1, workers=2, seed=0)

In [3]:
q = 0
print("クエリ:", reuters.categories(ids[q]), " ".join(docs[q].words[:25]))
for n, sim in model.dv.most_similar(q, topn=3):
    print(f"{sim:.2f}", reuters.categories(ids[n]), " ".join(docs[n].words[:25]))

クエリ: ['trade'] asian exporters fear damage from u s japan rift mounting trade friction between the u s and japan has raised fears among many of asia
0.78 ['trade'] japan in last ditch effort to avert tariffs senior japanese officials tomorrow open talks with american trade negotiators in a last ditch effort to avert
0.76 ['trade'] japan concerned at ec trade threats minister japan is disappointed at the recent threats of trade retaliation from the european community ec just as the
0.74 ['trade'] u s may end additional sanctions against japan the united states may lift an additional mln dlrs in trade sanctions against japan later this month


In [4]:
# 学習に使っていない新しい文書のベクトルを推論する
vec = model.infer_vector("oil prices rose after opec agreed to cut production".split(), epochs=50)
for n, sim in model.dv.most_similar([vec], topn=3):
    print(f"{sim:.2f}", reuters.categories(ids[n]), " ".join(docs[n].words[:15]))

0.64 ['trade'] canada february trade surplus billion dlrs after january mln dlrs surplus canada february trade surplus
0.63 ['ipi'] french industrial production fell pct in july august from june insee data french industrial production
0.63 ['earn'] rexham corp lt rxh qtr net shr cts vs cts net vs sales mln vs


- `dm=1` が PV-DM (文脈語 + 文書ベクトルで予測)、`dm=0` が PV-DBOW (文書ベクトルだけで単語を予測)。
- 現在は事前学習済みの文埋め込みモデルのほうが手軽で高精度なことが多い。ドメイン特化の大量文書がある場合の選択肢。